# 稀疏态演化示例

本教程展示 SparseState 如何随算子操作演化，并进一步演示 QFT 和 QRAM 等组合算子的使用。

In [1]:
import pysparq as ps

ps.System.clear()

# 创建 2 比特寄存器
ps.System.add_register("q", ps.UnsignedInteger, 2)

# 初始状态
state = ps.SparseState()

print("初始状态:")
ps.pprint(state)
print(f"\n基态数量: {state.size()}")

初始状态:
StatePrint (mode=Detail)
|(0)q : UInt2 | 
1.000000+0.000000i  q=|0>


基态数量: 1


## Hadamard 变换

Hadamard 创建量子叠加态。

In [2]:
# Hadamard_Int: 对指定数量的量子比特应用 Hadamard
ps.Hadamard_Int("q", 1)(state)

print("Hadamard_Int(q, 1) 后:")
ps.pprint(state)
print(f"\n基态数量: {state.size()}")

Hadamard_Int(q, 1) 后:
StatePrint (mode=Detail)
|(0)q : UInt2 | 
0.707107+0.000000i  q=|0>
0.707107+0.000000i  q=|1>


基态数量: 2


In [3]:
# Hadamard_Int_Full: 对所有量子比特应用完整 Hadamard
ps.Hadamard_Int_Full("q")(state)

print("Hadamard_Int_Full(q) 后:")
ps.pprint(state)
print(f"\n基态数量: {state.size()}")

Hadamard_Int_Full(q) 后:
StatePrint (mode=Detail)
|(0)q : UInt2 | 
0.707107+0.000000i  q=|0>
0.707107+0.000000i  q=|2>


基态数量: 2


## 状态打印模式

StatePrint 支持多种显示模式。

In [4]:
print("Default 模式:")
print(ps.StatePrint(state, mode=ps.StatePrintDisplay.Default))

Default 模式:
StatePrint (mode=Default)
0.707107+0.000000i |0>
0.707107+0.000000i |2>



In [5]:
print("Binary 模式:")
ps.pprint(state, mode=ps.StatePrintDisplay.Binary))

SyntaxError: unmatched ')' (3785520889.py, line 2)

In [6]:
print("Prob 模式:")
ps.pprint(state, mode=ps.StatePrintDisplay.Prob))

SyntaxError: unmatched ')' (3930040438.py, line 2)

## 算术操作演化

In [7]:
ps.System.clear()

# 创建寄存器
ps.System.add_register("a", ps.UnsignedInteger, 2)
ps.System.add_register("b", ps.UnsignedInteger, 2)
ps.System.add_register("sum", ps.UnsignedInteger, 2)

state = ps.SparseState()

# 初始化
ps.Init_Unsafe("a", 1)(state)
ps.Init_Unsafe("b", 2)(state)

print("初始状态:")
ps.pprint(state)

初始状态:
StatePrint (mode=Detail)
|(0)a : UInt2 | |(1)b : UInt2 | |(2)sum : UInt2 | 
1.000000+0.000000i  a=|1> b=|2> sum=|0>



In [8]:
# 加法: sum ^= a + b
ps.Add_UInt_UInt("a", "b", "sum")(state)

print("Add_UInt_UInt 后:")
ps.pprint(state)
# sum = 0 ^ (1 + 2) = 3

Add_UInt_UInt 后:
StatePrint (mode=Detail)
|(0)a : UInt2 | |(1)b : UInt2 | |(2)sum : UInt2 | 
1.000000+0.000000i  a=|1> b=|2> sum=|3>



In [9]:
# 再次应用 = 撤销（XOR 机制）
ps.Add_UInt_UInt("a", "b", "sum")(state)

print("再次 Add_UInt_UInt 后:")
ps.pprint(state)
# sum = 3 ^ 3 = 0

再次 Add_UInt_UInt 后:
StatePrint (mode=Detail)
|(0)a : UInt2 | |(1)b : UInt2 | |(2)sum : UInt2 | 
1.000000+0.000000i  a=|1> b=|2> sum=|0>



## 叠加态上的算术

In [10]:
ps.System.clear()

ps.System.add_register("x", ps.UnsignedInteger, 2)
ps.System.add_register("y", ps.UnsignedInteger, 2)

state = ps.SparseState()

# x 处于叠加态
ps.Hadamard_Int_Full("x")(state)

# y 初始化为常数
ps.Init_Unsafe("y", 1)(state)

print("初始叠加态:")
ps.pprint(state)

初始叠加态:
StatePrint (mode=Detail)
|(0)x : UInt2 | |(1)y : UInt2 | 
0.500000+0.000000i  x=|0> y=|1>
0.500000+0.000000i  x=|1> y=|1>
0.500000+0.000000i  x=|2> y=|1>
0.500000+0.000000i  x=|3> y=|1>



In [11]:
# 内置加法: y += x（每个分支独立计算）
ps.Add_UInt_UInt_InPlace("x", "y")(state)

print("Add_UInt_UInt_InPlace 后:")
ps.pprint(state)
# 每个分支: y = 1 + x

Add_UInt_UInt_InPlace 后:
StatePrint (mode=Detail)
|(0)x : UInt2 | |(1)y : UInt2 | 
0.500000+0.000000i  x=|0> y=|1>
0.500000+0.000000i  x=|1> y=|2>
0.500000+0.000000i  x=|2> y=|3>
0.500000+0.000000i  x=|3> y=|0>



In [12]:
# 撤销
ps.Add_UInt_UInt_InPlace("x", "y").dag(state)

print("dagger 后:")
ps.pprint(state)

dagger 后:
StatePrint (mode=Detail)
|(0)x : UInt2 | |(1)y : UInt2 | 
0.500000+0.000000i  x=|0> y=|1>
0.500000+0.000000i  x=|1> y=|1>
0.500000+0.000000i  x=|2> y=|1>
0.500000+0.000000i  x=|3> y=|1>



## 访问基态数据

In [13]:
# 遍历所有基态
print("遍历基态:")
for i, system in enumerate(state.basis_states):
    x_id = ps.System.get_id("x")
    y_id = ps.System.get_id("y")
    
    x_val = system.get(x_id).value
    y_val = system.get(y_id).value
    amp = system.amplitude
    
    print(f"  基态 {i}: x={x_val}, y={y_val}, 振幅={amp}")

遍历基态:
  基态 0: x=0, y=1, 振幅=(0.5+0j)
  基态 1: x=1, y=1, 振幅=(0.5+0j)
  基态 2: x=2, y=1, 振幅=(0.5+0j)
  基态 3: x=3, y=1, 振幅=(0.5+0j)


## QRAM 数据加载

在叠加态下通过 QRAM 批量加载经典数据：

```python
import numpy as np

ps.System.clear()
n_addr, n_data = 3, 4
ps.System.add_register("addr", ps.UnsignedInteger, n_addr)
ps.System.add_register("data", ps.UnsignedInteger, n_data)

state = ps.SparseState()

# 经典数据（8 个内存位置）
memory = np.array([1, 3, 5, 7, 2, 4, 6, 8], dtype=np.uint64)

# 对地址寄存器叠加 → 同时查询所有地址
ps.Hadamard_Int("addr")(state)

# QRAM 加载：data = memory[addr]
qram = ps.QRAMCircuit_qutrit(n_addr, n_data, memory)
ps.QRAMLoad(qram, "addr", "data")(state)

# 状态包含所有 (addr, memory[addr]) 对的振幅
print(ps.StatePrint()(state))
print(f"\n基态数量: {state.size()}")
```

QRAM 加载后，状态为所有 :math:`|addr\rangle \otimes |memory[addr]\rangle` 的叠加，基态数等于地址空间大小。

## QFT 变换

```python
ps.System.clear()
ps.System.add_register("reg", ps.UnsignedInteger, 3)
state = ps.SparseState()

ps.Init_Unsafe("reg", 1)(state)
print("初始:")
print(ps.StatePrint()(state))

ps.QFT("reg")(state)
print("QFT 后:")
print(ps.StatePrint()(state))

ps.inverseQFT("reg")(state)
print("逆 QFT 后:")
print(ps.StatePrint()(state))  # 恢复到 |1⟩
```

## 总结

- SparseState 只存储非零基态，基态数量随叠加增长
- Hadamard 创建叠加态
- 算术操作作用于每个基态（稀疏遍历）
- SelfAdjointOperator 两次应用恢复原值，BaseOperator 用 dag() 撤销
- QRAM 在叠加态下实现经典数据的批量查询
- QFT / inverseQFT 用于相位估计类算法